# Cookbook
Copy-paste analysis recipes using pandas. Each section is self-contained.

**Prerequisite:** Run any earlier notebook first to create a cached parquet file.

In [ ]:
from vulnflow import VulnDataset

ds = VulnDataset.from_parquet("../data/recent_cache.parquet")
df = ds.to_pandas()
print(f"{len(df):,} CVEs  |  {len(df.columns)} columns")
df.columns.tolist()

---
## Filter by Severity

In [ ]:
critical = df[df["cvss_base_score"] >= 9.0]
high = df[df["cvss_base_severity"] == "HIGH"]
print(f"Critical (>=9.0): {len(critical):,}")
print(f"High: {len(high):,}")
critical[["cve_id", "description", "cvss_base_score"]].head(10)

---
## Filter by Year

In [ ]:
df["year"] = df["cve_id"].str[4:8].astype(int)
df_2024 = df[df["year"] == 2024]
print(f"{len(df_2024):,} CVEs from 2024")

---
## Attack Vector Analysis

In [ ]:
# Attack vector distribution
df["cvss_attack_vector"].value_counts()

In [ ]:
# Network-accessible, no auth required, critical severity
remotely_exploitable = df[
    (df["cvss_attack_vector"] == "NETWORK")
    & (df["cvss_privileges_required"] == "NONE")
    & (df["cvss_user_interaction"] == "NONE")
    & (df["cvss_base_score"] >= 9.0)
]
print(f"{len(remotely_exploitable):,} remotely exploitable critical CVEs")
remotely_exploitable[["cve_id", "cvss_base_score", "description"]].head(5)

---
## CVSS Impact Analysis

In [ ]:
# CIA impact breakdown
for metric in ["cvss_confidentiality", "cvss_integrity", "cvss_availability"]:
    print(f"\n{metric}:")
    print(df[metric].value_counts().to_string())

---
## Transforms: CWE, CPE, Events
Run default transforms to extract CWE IDs, CPE vendor/product, and CVD event timestamps.

In [ ]:
ds.transform(quiet=True)
df = ds.to_pandas()
print(f"{len(df.columns)} columns after transforms")

### Top CWEs

In [ ]:
df["cwe_id"].value_counts().head(10)

### Top Vendors and Products (CPE)

In [ ]:
print("Top 10 vendors:")
print(df["cpe_vendor"].value_counts().head(10).to_string())

print("\nTop 10 products:")
print(df["cpe_product"].value_counts().head(10).to_string())

### Vendor Deep-Dive

In [ ]:
vendor = "microsoft"  # change to any vendor
vendor_df = df[df["cpe_vendor"] == vendor]
print(f"{len(vendor_df):,} CVEs for {vendor}")
print(f"\nSeverity:")
print(vendor_df["cvss_base_severity"].value_counts().to_string())
print(f"\nTop products:")
print(vendor_df["cpe_product"].value_counts().head(5).to_string())

---
## EPSS Scores
`CacheBuilder` enriches with EPSS by default. Columns: `epss_probability`, `epss_percentile`, `epss_date`, `epss_model`.

In [ ]:
scored = df[df["epss_probability"].notna()]
print(f"{len(scored):,} CVEs with EPSS scores")
print(f"Mean probability: {scored['epss_probability'].mean():.4f}")
print(f"Median probability: {scored['epss_probability'].median():.4f}")

In [ ]:
# High exploitation probability (>50%)
high_epss = df[df["epss_probability"] > 0.5]
print(f"{len(high_epss):,} CVEs with >50% exploitation probability")
high_epss[["cve_id", "epss_probability", "epss_percentile", "cvss_base_score"]].head(10)

In [ ]:
# EPSS vs CVSS: high EPSS but low CVSS (prioritization mismatches)
mismatch = df[
    (df["epss_probability"] > 0.5)
    & (df["cvss_base_score"] < 7.0)
]
print(f"{len(mismatch):,} CVEs with high EPSS but low CVSS")
mismatch[["cve_id", "epss_probability", "cvss_base_score", "cvss_base_severity"]].head(10)

---
## KEV (Known Exploited Vulnerabilities)
`CacheBuilder` enriches with CISA KEV by default. Columns: `is_kev`, `kev_date_added`, `kev_due_date`, `kev_vendor`, `kev_product`, `kev_ransomware`.

In [ ]:
kev = df[df["is_kev"] == True]
print(f"{len(kev):,} CVEs in CISA KEV catalog")
kev[["cve_id", "kev_date_added", "kev_due_date", "kev_vendor", "kev_product"]].head(10)

In [ ]:
# KEV entries linked to ransomware campaigns
ransomware = kev[kev["kev_ransomware"] == True]
print(f"{len(ransomware):,} KEV entries with known ransomware use")
ransomware[["cve_id", "kev_vendor", "kev_product", "kev_date_added"]].head(10)

In [ ]:
# KEV by severity — are all KEV entries critical/high?
print("KEV entries by CVSS severity:")
print(kev["cvss_base_severity"].value_counts().to_string())

---
## CWE Enrichment
*Requires CWEEnricher (fetches CWE database from NVD). Currently only `cwe_id` is available from NVD data; `cwe_name` and `cwe_category_name` are populated after enrichment.*

In [ ]:
# TODO: CWE enrichment adds weakness names and categories
#
# from vulnflow.transforms.enrichers.cwe import CWEEnricher
# ds.transform([CWEEnricher()])
# df = ds.to_pandas()
# df[["cwe_id", "cwe_name", "cwe_category_name"]].drop_duplicates().head(10)

---
## Metadata & Tags
*Not yet implemented. These columns are reserved for user-defined metadata and auto-generated tags.*

In [ ]:
# TODO: metadata and tags are placeholder columns
#
# metadata: dict per CVE for user-defined key-value pairs
# tags: dict per CVE for auto-generated classification tags

---
## Export to CSV

In [ ]:
df.to_csv("../data/export.csv", index=False)
print(f"Exported {len(df):,} records to data/export.csv")